### 멀티모달 RAG
- 01.multimodal 에서 추출한 텍스트와 이미지요약본을 VectorDB에 임베딩하고 

    사용자의 질문에 맞춰 관련 컨텍스트를 검색(Retrival)한뒤 LLM을 이용해서 최종 답변을 생성(Generation)

In [1]:
import os
import fitz
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

In [2]:
# 환경변수 로드 및 openai 클라이언트 생성
load_dotenv(override=True)
client = OpenAI()

In [29]:
# 벡터 Db 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db')
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv('OPENAI_API_KEY'),
    model_name = 'text-embedding-3-small'
)
if chroma_client.count_collections() != 0:
    chroma_client.delete_collection('multimodal_rag')
collection = chroma_client.get_or_create_collection(name='multimodal_rag',embedding_function=openai_ef)
print(f'설정완료 현재 컬렉션의 크기 : {collection.count()}')

설정완료 현재 컬렉션의 크기 : 0


### 데이터 임베딩(텍스트 + 이미지 요약본)
- pdf의 텍스트와 이미지 요약본을 모두 VectorDB에 삽입(동일한 벡터공간에 텍스트와 이미지 요약본이 함께 존재)

In [30]:
import base64
if collection.count() == 0:
    pdf_path = 'sample_paper.pdf'
    doc = fitz.open(pdf_path)
    documents = []
    metatdatas = []    
    ids = []
    
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # 텍스트 추출
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metatdatas.append({'page':page_num+1,'type':'text'})
            ids.append(f'text_page_{page_num+1}')
        # 이미지 요약 후 추출
        image_list = page.get_images(full=True)
        if image_list:
            xref = image_list[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            if ext.lower() == 'jb2': ext = 'jpeg'
            mime_type = f'image/{ext}' if ext.lower() != "jpg" else 'image/jpeg'
            base64_image = base64.b64encode(image_bytes).decode('utf-8')
            try:
                response = client.chat.completions.create(
                    model = 'gpt-5.4-nano',
                    messages=[
                        {'role':'user', 
                        'content': [
                            {'type':'text', 'text':'이 이미지에 대한 상세한 설명을 작성해 주세요. 전문적이고 구체적으로 분석해야 합니다.'},
                            {'type':'image_url', 'image_url':{
                                'url':f'data:{mime_type};base64,{base64_image}'
                                }
                            }
                        ]
                        }
                    ],
                    # max_completion_tokens=300
                )
                summary = response.choices[0].message.content
                documents.append(summary)
                metatdatas.append({
                    'type':'image_summary','page':page_num+1
                })
                ids.append(f'image_summary_page_{page_num+1}')
            except Exception as e:
                pass
    if documents:
        collection.add(documents=documents,metadatas=metatdatas,ids=ids)
        print('VectorDB에 데이터 임베딩 및 저장 완료')
else:
    print(f'vectorDB에 이미 {collection.count()}개 데이터가 존재합니다')


VectorDB에 데이터 임베딩 및 저장 완료


### 질문을 통해 RAG 검색하기

In [31]:
query = '논문에 나온 트랜스포커 아키텍처 다이어그램(인코더-디코더 그림)에 대해 설명해주세요'
results = collection.query(
    query_texts=[query],
    n_results=1
)
retrieved_docs = results['documents'][0][0]
retrieved_metas = results['metadatas'][0][0]
print(retrieved_docs)
print(retrieved_metas)

아래 이미지는 **Transformer(인코더–디코더) 기반의 시퀀스-투-시퀀스 모델** 동작을 한 장의 도식으로 설명한 것입니다. 특히 **인코더는 입력 문맥을 인코딩**하고, **디코더는 마스킹된 자기회귀 방식으로 다음 토큰을 생성**하며, 마지막에 **Softmax로 출력 확률**을 산출하는 흐름을 강조합니다.

---

## 1) 전체 구조: 인코더(좌측) + 디코더(우측)
도식은 크게 두 블록으로 나뉩니다.

- **좌측 큰 박스: Encoder**
  - 입력 시퀀스를 받아 반복되는 레이어 스택을 통과시킵니다.
  - 레이어 수는 **N×** 로 표시되어 여러 번(예: N=6) 반복됩니다.
- **우측 큰 박스: Decoder**
  - 이전에 생성된 토큰(오른쪽으로 shift된 출력)을 기반으로 다음 토큰을 예측합니다.
  - 역시 **N×** 로 표시된 레이어 스택을 통과합니다.
- 두 블록 사이에는 **디코더의 “Multi-Head Attention”이 인코더 출력(메모리/키-값)**을 참고하는 구조(가로 방향 연결)가 존재합니다.

---

## 2) 입력/출력 임베딩 및 위치 정보(Positional Encoding)
### (1) Input Embedding (하단 좌측)
- **Inputs** 레이블 아래에 있는 **Input Embedding** 박스가 있습니다.
- 이는 토큰(단어/서브워드)을 **벡터(임베딩 차원)** 로 변환하는 단계입니다.

### (2) Positional Encoding (하단 좌측·우측)
- Embedding의 출력과 함께 **Positional Encoding**이 더해집니다.
- Transformer는 순서 정보를 내부적으로 처리하지 않기 때문에, **토큰 위치(순서)** 를 벡터에 반영하기 위해 사용됩니다.
- 도식에서는 **"+"(더하기) 기호**로 임베딩과 위치 인코딩이 결합되는 흐름이 시각화되어 있습니다.

### (3) Output Embedding (하단 우측)
- 디코더에는 **Output 